In [48]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [49]:
from google.colab import drive

drive.mount('/content/drive')
!cp -r -n /content/drive/MyDrive/hustleflow_data/ /content/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [50]:
train_csv = '/content/hustleflow_data/train_raw.csv'
test_csv = '/content/hustleflow_data/test_raw.csv'

In [51]:
train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)

In [52]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 960 entries, 0 to 959
Data columns (total 28 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   EmpNumber                     960 non-null    object
 1   Age                           960 non-null    int64 
 2   Gender                        960 non-null    object
 3   EducationBackground           960 non-null    object
 4   MaritalStatus                 960 non-null    object
 5   EmpDepartment                 960 non-null    object
 6   EmpJobRole                    960 non-null    object
 7   BusinessTravelFrequency       960 non-null    object
 8   DistanceFromHome              960 non-null    int64 
 9   EmpEducationLevel             960 non-null    int64 
 10  EmpEnvironmentSatisfaction    960 non-null    int64 
 11  EmpHourlyRate                 960 non-null    int64 
 12  EmpJobInvolvement             960 non-null    int64 
 13  EmpJobLevel         

Preprocess

In [53]:
X_train = train_df.drop(columns=['PerformanceRating', 'EmpNumber'])
y_train = train_df['PerformanceRating']
X_test = test_df.drop(columns=['PerformanceRating', 'EmpNumber'])
y_test = test_df['PerformanceRating']

In [54]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

cat_features = X_train.select_dtypes(include=['object']).columns
num_features = X_train.select_dtypes(include=np.number).columns

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
    ('num', StandardScaler(), num_features)
])

Model pipeline

In [55]:
from sklearn.svm import SVC
from imblearn.pipeline import Pipeline
from imblearn.combine import SMOTETomek

model = Pipeline([
    ('preprocess', preprocessor),
    ('sampler', SMOTETomek(random_state=42)),
    ('svc', SVC(probability=True, random_state=42))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)

In [56]:
from sklearn.metrics import classification_report, roc_auc_score

print('ROC AUC score: ', roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted'))
print('\nClassification report:')
print(classification_report(y_test, y_pred))

ROC AUC score:  0.9064018766102391

Classification report:
              precision    recall  f1-score   support

           0       0.63      0.67      0.65        39
           1       0.90      0.89      0.89       175
           2       0.59      0.62      0.60        26

    accuracy                           0.82       240
   macro avg       0.71      0.72      0.72       240
weighted avg       0.82      0.82      0.82       240



Hyperparameter Tuning

In [57]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, loguniform

param_distributions = [
    {
        "svc__kernel": ["linear"],
        "svc__C": loguniform(1e-2, 1e2),
        "svc__class_weight": ["balanced"]
    },
    {
        "svc__kernel": ["rbf"],
        "svc__C": loguniform(1e-2, 1e3),
        "svc__gamma": loguniform(1e-4, 1e1),
        "svc__class_weight": ["balanced"]
    },
    {
        "svc__kernel": ["poly"],
        "svc__C": loguniform(1e-2, 1e3),
        "svc__gamma": loguniform(1e-4, 1e1),
        "svc__class_weight": ["balanced"],
        'svc__degree': [2, 3, 4]
    }
]

grid = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_distributions,
    scoring="f1_macro",
    cv=10,
    n_jobs=-1,
    n_iter=50,
    verbose=0
)

grid.fit(X_train, y_train)
y_pred_tuned = grid.predict(X_test)
y_pred_proba_tuned = grid.predict_proba(X_test)

In [58]:
print('Best params: ', grid.best_params_)
print('Best cross-validation score: ', grid.best_score_)

Best params:  {'svc__C': np.float64(294.5074990607953), 'svc__class_weight': 'balanced', 'svc__degree': 2, 'svc__gamma': np.float64(0.001276536884670072), 'svc__kernel': 'poly'}
Best cross-validation score:  0.7156478115457396


In [59]:
from sklearn.metrics import classification_report, roc_auc_score

print('ROC AUC score: ', roc_auc_score(y_test, y_pred_proba_tuned, multi_class='ovr', average='weighted'))
print('\nClassification report:')
print(classification_report(y_test, y_pred_tuned))

ROC AUC score:  0.8979598487910317

Classification report:
              precision    recall  f1-score   support

           0       0.54      0.72      0.62        39
           1       0.90      0.81      0.85       175
           2       0.59      0.73      0.66        26

    accuracy                           0.78       240
   macro avg       0.68      0.75      0.71       240
weighted avg       0.81      0.78      0.79       240

